# BMKMANU S3000 - exp101 100M

## Import packages

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

In [6]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
sys.path.append(os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(''))), 'BMK_S1000/harpy'))
from utils.in_tissue import add_in_tissue
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _bmkmanu import bmkmanu

## Create sdata

In [7]:
dataset_id = 'exp101'

path = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/BSTMAtrix'
bc_path = f'e:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/bc2bc/{dataset_id}.bc'
config_prefix = 'exp101_100M_11Aug2025'
technology = 'S3000'
output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy/sdata.zarr'

pipeline_name = 'BSTMAtrix'
species = 'mouse'
tissue_annotation_file = f'E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/tissue_annotation/tissue.geojson'


In [8]:
sdata = bmkmanu(
    path=path, 
    dataset_id=dataset_id,
    filtered_counts=False,
    add_in_tissue_column=True,
    levels=None, 
    technology=technology,
    bc_path=bc_path,
    output=output_path
)
sdata

2025-08-25 15:40:45,044 - harpy.io._spatial_data - WARNING - dims parameter is equal to ['c', 'y', 'x'], but will be ignored when reading in images from a file
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-08-25 15:40:54,224 - harpy.image._manager - INFO - Writing results to layer 'exp101_he'
2025-08-25 15:45:40,279 - harpy.image._manager - INFO - Writing results to layer 'exp101_level_1_labels'
2025-08-25 15:46:55,022 - harpy.image._manager - INFO - Writing results to layer 'exp101_level_2_labels'
2025-08-25 15:47:37,227 - harpy.image._manager - INFO - Writing results to layer 'exp101_level_3_labels'
2025-08-25 15:48:08,579 - harpy.image._manager - INFO - Writing results to layer 'exp101_level_4_labels'
2025-08-25 15:48:36,312 - harpy.image._manager - INFO - Writing results to layer 'exp101_level_5_labels'
2025-08-25 15:49

INFO     The Zarr backing store has been changed from None the new file path:                                      
         E:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp101\subsampled_100M\harpy\sdata.zarr               


version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: E:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\Exp101\subsampled_100M\harpy\sdata.zarr
├── Images
│     └── 'exp101_he': DataTree[cyx] (3, 20000, 19954), (3, 10000, 9977), (3, 5000, 4988), (3, 2500, 2494), (3, 1250, 1247)
├── Labels
│     ├── 'exp101_level_1_labels': DataArray[yx] (19996, 19949)
│     ├── 'exp101_level_2_labels': DataArray[yx] (20013, 19967)
│     ├── 'exp101_level_3_labels': DataArray[yx] (20031, 19986)
│     ├── 'exp101_level_4_labels': DataArray[yx] (20048, 20005)
│     ├── 'exp101_level_5_labels': DataArray[yx] (20066, 20025)
│     ├── 'exp101_level_6_labels': DataArray[yx] (20084, 20044)
│     ├── 'exp101_level_7_labels': DataArray[yx] (20102, 20063)
│     ├── 'exp101_level_9_labels': DataArray[yx] (20137, 20101)
│     └── 'exp101_level_18_labels': DataArray[yx] (20296, 20273)
├── Shapes
│     ├── 'exp101_level_1': GeoDataFrame shape: (3206235, 1) (2D shapes)
│     ├── 'exp101_level_2': GeoDataFrame shape: (6177

In [9]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

BSTMAtrix levels: ['level_1', 'level_18', 'level_2', 'level_3', 'level_4', 'level_5', 'level_6', 'level_7', 'level_9']


## Specify plotting parameters

In [10]:
DPI = 300
MAX_DIM_FOR_PLOTS = 5000

HE_image = f'{dataset_id}_he'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale2
figsize: (16.626666666666665, 16.666666666666668)
plots_path: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots


## Plot H&E

In [11]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\HE.png


## Plot BSTMAtrix levels and tissue annotation

In [12]:
for level in levels:
    gc.collect()
    with matplotlib.rc_context({'legend.markerscale': 10}):
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} {level} - {pipeline_name} tissue annotation"
        
        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=0.1, contour_px=5, outline_alpha=1, color='in_tissue', scale=best_scale
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=False,
            legend_fontsize=rel_fontsize(fig, 0.015), 
            legend_fontweight='bold', 
            legend_loc='right margin',
            na_in_legend=False,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\BSTMAtrix_tissue_annotation_level_1.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\BSTMAtrix_tissue_annotation_level_18.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


...

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\BSTMAtrix_tissue_annotation_level_6.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\BSTMAtrix_tissue_annotation_level_7.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\BSTMAtrix_tissue_annotation_level_9.png


## Calculate pixel size

In [13]:
# Determine pixel size in microns
he_shape = hp.im._image._get_spatial_element(sdata, f'{dataset_id}_he').shape

he_width = he_shape[2]
he_height = he_shape[1]

microns_per_pixel_x = 6800 / he_shape[2]
microns_per_pixel_y = 6800 / he_shape[1]
print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to summise what the pixel size is. This was recommended by BMKGENE and should be good enough, but there is definitely the possibility that the capture area is not exactly 6.8 mm.

Pixel size: 0.341 µm (X), 0.340 µm (Y)
Pixel size: 0.340 µm


## Create bounding box for total capture area

In [14]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_level_1'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

In [15]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_level_1', color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.03)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' do disable this    
         behaviour.                                                                                                
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\capture_area_bbox.png


In [16]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size. However, it should be close.

Area of capture area: 46.21 mm²


In [17]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size

Dimensions of capture area bbox: 0.68 cm x 0.68 cm


## Add tissue annotation

In [18]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-08-25 16:07:52,273 - harpy.image._manager - INFO - Writing results to layer 'exp101_tissue_mask'


In [19]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_tissue_annotation.png


In [20]:
# Determine which spots are in tissue annotation
for level in levels:
    gc.collect()
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

2025-08-25 16:10:30,830 - harpy.utils._io - WARNING - layer with name 'level_1' already exists. Overwriting...
2025-08-25 16:12:17,380 - harpy.utils._io - WARNING - layer with name 'level_18' already exists. Overwriting...
2025-08-25 16:13:00,072 - harpy.utils._io - WARNING - layer with name 'level_2' already exists. Overwriting...
2025-08-25 16:13:43,613 - harpy.utils._io - WARNING - layer with name 'level_3' already exists. Overwriting...
2025-08-25 16:14:24,905 - harpy.utils._io - WARNING - layer with name 'level_4' already exists. Overwriting...
2025-08-25 16:15:05,431 - harpy.utils._io - WARNING - layer with name 'level_5' already exists. Overwriting...
2025-08-25 16:15:44,550 - harpy.utils._io - WARNING - layer with name 'level_6' already exists. Overwriting...
2025-08-25 16:16:25,152 - harpy.utils._io - WARNING - layer with name 'level_7' already exists. Overwriting...
2025-08-25 16:17:00,955 - harpy.utils._io - WARNING - layer with name 'level_9' already exists. Overwriting...


In [21]:
# Check spots in tissue
for level in levels:
    gc.collect()
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = "Tissue annotation spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue_annotation', table_layer=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.02)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'qupath_annotation_spots_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_annotation_spots_level_1.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_annotation_spots_level_18.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


...

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_annotation_spots_level_6.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_annotation_spots_level_7.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\qupath_annotation_spots_level_9.png


In [22]:
# Total number of spots in table
n_spots = len(sdata.tables['level_1'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 3206235


In [23]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables['level_1'].obs[sdata.tables['level_1'].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

Number of spots in table with at least one UMI: 2388173


In [24]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

Number of spots in tissue: 963658


In [25]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = sdata.tables['level_1'].obs['in_tissue_annotation'] == 1
umis_mask = np.array(sdata.tables['level_1'].X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

Number of spots in tissue with at least one UMI: 933534


In [26]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 13.01 mm²


## Capture effeciency metrics

In [27]:
# Total number of genes
print(f"Total number of genes: {(sdata.tables['level_1'].X.sum(axis=0) > 0).sum()}")

Total number of genes: 26375


In [28]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 25977


In [29]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['level_1'].X.sum()}")

Total number of unique UMIs: 45372144.0


In [30]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum()}")

Number of unique UMIs in tissue: 39136532.0


In [31]:
umi_gene = pd.read_csv(os.path.join(path, config_prefix, 'umi_gene.tsv.gz'), sep='\t', header=None)
umi_gene

,0,1,2,3
0,V2Bbc1_1-V2Abc2_248-V2Abc3_279,CGACTGGGTTTT,ENSMUSG00000042675,5
1,V2Bbc1_1-V2Abc2_248-V2Abc3_279,TTCTGATGGGTG,ENSMUSG00000031633,1
2,V2Bbc1_1-V2Abc2_248-V2Abc3_279,GATATAGGTCGT,ENSMUSG00000025716,1
3,V2Bbc1_1-V2Abc2_248-V2Abc3_279,TTTTGATGCCGT,ENSMUSG00000026980,2
4,V2Bbc1_1-V2Abc2_248-V2Abc3_279,GAATACGAGTAT,ENSMUSG00000079264,1
...,...,...,...,...
49272320,V2Bbc1_58-V2Bbc2_196-V2Bbc3_8,ATTGGAAATCG,ENSMUSG00000064341,1
49272321,V2Bbc1_58-V2Bbc2_196-V2Bbc3_8,CCCAGGATAAT,ENSMUSG00000051359,1
49272322,V2Bbc1_58-V2Bbc2_196-V2Bbc3_8,AGCATCAAACC,ENSMUSG00000034845,1
49272323,V2Bbc1_58-V2Bbc2_196-V2Bbc3_8,TCGGCTATAAA,ENSMUSG00000025967,1


In [32]:
# Total number of UMIs
print("Total number of UMIs:", umi_gene[3].sum())

Total number of UMIs: 69717755


In [33]:
# Total number of unique barcodes
print("Total number of unique barcodes:", umi_gene[0].nunique())

Total number of unique barcodes: 2549918


In [34]:
barcode_pos = pd.read_csv(os.path.join(path, config_prefix, 'barcode_pos.tsv'), sep='\t', header=None)
barcode_pos

,0,1,2,3,4
0,V2Abc1_279-V2Bbc2_41-V2Abc3_72,0,0,0,11
1,V2Bbc1_23-V2Abc2_125-V2Abc3_349,0,0,0,13
2,V2Abc1_195-V2Bbc2_322-V2Bbc3_193,0,0,0,17
3,V2Bbc1_310-V2Bbc2_279-V2Bbc3_329,0,0,0,24
4,V2Abc1_131-V2Abc2_183-V2Abc3_379,0,0,0,27
...,...,...,...,...,...
2842105,V2Bbc1_276-V2Bbc2_141-V2Bbc3_128,42,41,50,38
2842106,V2Abc1_233-V2Abc2_273-V2Abc3_308,42,41,50,39
2842107,V2Abc1_77-V2Abc2_284-V2Bbc3_178,42,41,50,40
2842108,V2Bbc1_286-V2Bbc2_78-V2Bbc3_230,42,41,50,41


In [35]:
features = pd.read_csv(os.path.join(path, config_prefix, 'features.tsv'), sep='\t', header=None)
features

,0,1,2
0,ENSMUSG00000051951,Xkr4,Gene Expression
1,ENSMUSG00000089699,Gm1992,Gene Expression
2,ENSMUSG00000102331,Gm19938,Gene Expression
3,ENSMUSG00000102343,Gm37381,Gene Expression
4,ENSMUSG00000025900,Rp1,Gene Expression
...,...,...,...
32280,ENSMUSG00000095523,AC124606.1,Gene Expression
32281,ENSMUSG00000095475,AC133095.2,Gene Expression
32282,ENSMUSG00000094855,AC133095.1,Gene Expression
32283,ENSMUSG00000095019,AC234645.1,Gene Expression


## Export unique UMI counts per gene

In [36]:
adata_in_tissue = sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

## QC

In [37]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    gc.collect()
    sdata = qc_metrics(
        sdata,
        level,
        species
    )

2025-08-25 16:29:20,347 - harpy.utils._io - WARNING - layer with name 'level_1' already exists. Overwriting...
2025-08-25 16:30:18,216 - harpy.utils._io - WARNING - layer with name 'level_18' already exists. Overwriting...
2025-08-25 16:31:05,533 - harpy.utils._io - WARNING - layer with name 'level_2' already exists. Overwriting...
2025-08-25 16:31:58,661 - harpy.utils._io - WARNING - layer with name 'level_3' already exists. Overwriting...
2025-08-25 16:33:00,607 - harpy.utils._io - WARNING - layer with name 'level_4' already exists. Overwriting...
2025-08-25 16:34:11,550 - harpy.utils._io - WARNING - layer with name 'level_5' already exists. Overwriting...
2025-08-25 16:35:17,448 - harpy.utils._io - WARNING - layer with name 'level_6' already exists. Overwriting...
2025-08-25 16:36:02,300 - harpy.utils._io - WARNING - layer with name 'level_7' already exists. Overwriting...
2025-08-25 16:36:52,034 - harpy.utils._io - WARNING - layer with name 'level_9' already exists. Overwriting...


In [38]:
# Plot QC metrics
for level in levels:
    gc.collect()
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()
        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"scanpy - {qc_metric}"

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, scale=best_scale, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
                
        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'scanpy_{qc_metric}_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\scanpy_n_genes_by_counts_level_1.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\scanpy_total_counts_level_1.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


...

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\scanpy_pct_counts_in_top_50_genes_level_9.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\scanpy_total_counts_mt_level_9.png


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-0.013574661..1.0].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/Exp101/subsampled_100M/harpy\plots\scanpy_pct_counts_mt_level_9.png
